# Poster Generation Model Comparison

This notebook compares **4 generated-poster models** against the **original DCORE dataset** using the CSV outputs.

### Models
- Gemini
- OpenAI
- Qwen3
- Qwen-ImageEdit

### What this analysis measures
1. Overall score distribution
2. Per-poster score improvement/degradation
3. Statistical significance of paired score changes
4. Feature-level changes across models
5. Issue-frequency changes
6. Score vs. visual/text feature relationships
7. Model consistency and variability
8. A compact final comparison table with research-useful findings

> The analysis treats the `score` already present in your CSV files as the evaluation score. It does **not** create a new scoring formula.


In [ ]:
# If using Google Colab, upload the five CSV files first.
# from google.colab import files
# files.upload()

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from pathlib import Path

from scipy.stats import wilcoxon, spearmanr

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")

print("Libraries loaded.")


In [ ]:
# =========================
# 1. FILE CONFIGURATION
# =========================

# Change BASE_DIR if your CSVs are stored somewhere else.
BASE_DIR = Path("/content")

FILES = {
    "Original": BASE_DIR / "poster_outputs_dataset33.csv",
    "Gemini": BASE_DIR / "poster_outputs_gemini_generatedv2.csv",
    "OpenAI": BASE_DIR / "poster_outputs_openai_generatedv2.csv",
    "Qwen3": BASE_DIR / "poster_outputs_qwen3_generatedv2.csv",
    "Qwen-ImageEdit": BASE_DIR / "poster_outputs_qwenimageedit_generatedv2.csv",
}

for model, path in FILES.items():
    print(f"{model:18} -> {path} | exists={path.exists()}")

missing = [str(p) for p in FILES.values() if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Missing CSV files. Upload them to Colab or update BASE_DIR/FILes.\n" +
        "\n".join(missing)
    )


In [ ]:
# =========================
# 2. LOAD + STANDARDIZE
# =========================

def extract_poster_id(filename):
    # Works with poster1.jpeg, poster10.png, poster10_gemini.png, etc.
    m = re.search(r"poster(\d+)", str(filename).lower())
    return int(m.group(1)) if m else np.nan

frames = []

for model, path in FILES.items():
    df = pd.read_csv(path)
    df["poster_id"] = df["file"].apply(extract_poster_id)
    df["model"] = model
    frames.append(df)

data = pd.concat(frames, ignore_index=True)

feature_cols = [
    "brightness", "contrast", "entropy", "edge_density",
    "hue_diversity", "lr_balance", "tb_balance",
    "whitespace", "saliency", "text_density", "word_count"
]

required_cols = ["file", "score", "issues", "poster_id", "model"] + feature_cols
missing_cols = [c for c in required_cols if c not in data.columns]

if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}")

data = data.sort_values(["poster_id", "model"]).reset_index(drop=True)

print("Shape:", data.shape)
print("Posters:", data["poster_id"].nunique())
print("Models:", data["model"].unique())
display(data.head())


In [ ]:
# =========================
# 3. DATA QUALITY CHECK
# =========================

quality = pd.DataFrame({
    "rows": data.groupby("model").size(),
    "unique_posters": data.groupby("model")["poster_id"].nunique(),
    "missing_score": data.groupby("model")["score"].apply(lambda s: s.isna().sum()),
    "missing_issues": data.groupby("model")["issues"].apply(lambda s: s.isna().sum()),
})

display(quality)

# Check that every model has the same poster IDs.
id_sets = {m: set(data.loc[data.model == m, "poster_id"]) for m in FILES}
reference_ids = id_sets["Original"]

for model, ids in id_sets.items():
    print(model, "matches original IDs:", ids == reference_ids)

assert all(ids == reference_ids for ids in id_sets.values()),     "Poster IDs do not match across CSVs. Paired analysis cannot be trusted until they are aligned."


## 4. Overall Score Comparison

The first view answers a simple question:

**How does the distribution of generated-poster scores compare with the original posters?**

We show mean, median, standard deviation, minimum, maximum, and quartiles rather than relying on one number.


In [ ]:
score_summary = (
    data.groupby("model")["score"]
    .agg(
        mean="mean",
        median="median",
        std="std",
        min="min",
        q25=lambda x: x.quantile(.25),
        q75=lambda x: x.quantile(.75),
        max="max"
    )
    .reindex(FILES.keys())
    .round(2)
)

display(score_summary)

plt.figure(figsize=(10, 5))
sns.boxplot(data=data, x="model", y="score", order=list(FILES.keys()))
sns.stripplot(data=data, x="model", y="score", order=list(FILES.keys()),
              color="black", alpha=.35, size=3)
plt.title("Poster Score Distribution: Original vs Generated Models")
plt.xlabel("")
plt.ylabel("Poster score")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


In [ ]:
# Mean score comparison
mean_scores = score_summary["mean"].sort_values(ascending=False)

plt.figure(figsize=(9, 5))
ax = sns.barplot(x=mean_scores.index, y=mean_scores.values)
plt.title("Mean Poster Score by Model")
plt.xlabel("")
plt.ylabel("Mean score")
plt.xticks(rotation=15)

for i, v in enumerate(mean_scores.values):
    ax.text(i, v + 0.5, f"{v:.2f}", ha="center")

plt.tight_layout()
plt.show()


## 5. Paired Improvement vs Original

Because every generated poster corresponds to an original poster, the most useful comparison is **paired**:

`generated score - original score`

This avoids treating the 33 posters as unrelated samples.

We calculate:
- mean change
- median change
- percentage of posters that improved
- percentage that declined
- Wilcoxon signed-rank test
- paired Cohen's *d*

The statistical test is supplementary evidence; the actual magnitude of change is usually more informative.


In [ ]:
original = (
    data[data.model == "Original"]
    .set_index("poster_id")
)

generated_models = ["Gemini", "OpenAI", "Qwen3", "Qwen-ImageEdit"]

paired_results = []

for model in generated_models:
    gen = data[data.model == model].set_index("poster_id")

    delta = gen["score"] - original["score"]
    delta = delta.dropna()

    try:
        stat, p = wilcoxon(delta)
    except ValueError:
        stat, p = np.nan, np.nan

    sd_delta = delta.std(ddof=1)
    cohens_d = delta.mean() / sd_delta if sd_delta != 0 else np.nan

    paired_results.append({
        "Model": model,
        "Mean change": delta.mean(),
        "Median change": delta.median(),
        "Improved": int((delta > 0).sum()),
        "Unchanged": int((delta == 0).sum()),
        "Declined": int((delta < 0).sum()),
        "Improved %": (delta > 0).mean() * 100,
        "Declined %": (delta < 0).mean() * 100,
        "Wilcoxon p": p,
        "Paired Cohen's d": cohens_d
    })

paired_results = pd.DataFrame(paired_results).set_index("Model").round(4)
display(paired_results)


In [ ]:
# Per-poster delta plot
delta_df = []

for model in generated_models:
    gen = data[data.model == model].set_index("poster_id")
    temp = pd.DataFrame({
        "poster_id": original.index,
        "delta": gen["score"] - original["score"],
        "model": model
    })
    delta_df.append(temp.reset_index(drop=True))

delta_df = pd.concat(delta_df, ignore_index=True)

plt.figure(figsize=(13, 6))
sns.boxplot(data=delta_df, x="model", y="delta", order=generated_models)
sns.stripplot(data=delta_df, x="model", y="delta", order=generated_models,
              color="black", alpha=.4, size=3)
plt.axhline(0, linestyle="--", linewidth=1)
plt.title("Paired Score Change Relative to Original")
plt.xlabel("")
plt.ylabel("Generated score − Original score")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


In [ ]:
# Distribution of paired score changes
plt.figure(figsize=(11, 6))
for model in generated_models:
    vals = delta_df.loc[delta_df["model"] == model, "delta"]
    sns.kdeplot(vals, label=model, fill=False)

plt.axvline(0, linestyle="--", linewidth=1)
plt.title("Distribution of Score Improvements / Declines")
plt.xlabel("Score change relative to original")
plt.ylabel("Density")
plt.legend()
plt.tight_layout()
plt.show()


## 6. Feature-Level Comparison

Score alone can hide *why* a model changed.

For every visual/text feature, this section reports the average feature value for each model and the mean change from the original dataset.

Useful examples:
- `contrast`: readability-related visual separation
- `edge_density`: visual complexity
- `whitespace`: layout breathing room
- `text_density` / `word_count`: information density
- `hue_diversity`: color variety
- `saliency`: visual attention concentration


In [ ]:
feature_means = (
    data.groupby("model")[feature_cols]
    .mean()
    .reindex(FILES.keys())
)

display(feature_means.round(4))


In [ ]:
# Mean feature delta relative to original
feature_delta_tables = {}

for model in generated_models:
    gen = data[data.model == model].set_index("poster_id")
    orig = original

    delta = (gen[feature_cols] - orig[feature_cols]).mean()
    feature_delta_tables[model] = delta

feature_delta = pd.DataFrame(feature_delta_tables).T
display(feature_delta.round(4))


In [ ]:
# Heatmap of feature changes
plt.figure(figsize=(14, 5))
sns.heatmap(feature_delta, annot=True, fmt=".3f", center=0, cmap="coolwarm")
plt.title("Mean Feature Change Relative to Original")
plt.xlabel("Feature")
plt.ylabel("Generated model")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# Feature distributions across all five groups
for feature in feature_cols:
    plt.figure(figsize=(9, 4.5))
    sns.boxplot(data=data, x="model", y=feature, order=list(FILES.keys()))
    plt.title(f"{feature}: Distribution Across Models")
    plt.xlabel("")
    plt.ylabel(feature)
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.show()


## 7. Which Features Are Associated With Higher Scores?

This is a descriptive analysis, not a causal claim.

We calculate Spearman correlation between `score` and each numeric feature **within each model**. Spearman is useful here because relationships do not have to be linear.

The strongest absolute correlations can identify features worth discussing in the research analysis.


In [ ]:
correlation_rows = []

for model in FILES.keys():
    subset = data[data.model == model]

    for feature in feature_cols:
        rho, p = spearmanr(subset[feature], subset["score"], nan_policy="omit")
        correlation_rows.append({
            "Model": model,
            "Feature": feature,
            "Spearman rho": rho,
            "p-value": p
        })

corr_df = pd.DataFrame(correlation_rows)

# Show strongest absolute associations for every model
for model in FILES.keys():
    print(f"\n=== {model} ===")
    display(
        corr_df[corr_df["Model"] == model]
        .assign(abs_rho=lambda x: x["Spearman rho"].abs())
        .sort_values("abs_rho", ascending=False)
        .drop(columns="abs_rho")
        .round(4)
    )


In [ ]:
# Correlation heatmap
corr_matrix = (
    corr_df
    .pivot(index="Feature", columns="Model", values="Spearman rho")
    .reindex(columns=list(FILES.keys()))
)

plt.figure(figsize=(11, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", center=0, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Spearman Correlation: Poster Score vs Features")
plt.xlabel("")
plt.ylabel("")
plt.tight_layout()
plt.show()


## 8. Issue Analysis

The `issues` column gives a second, interpretable view of model behavior.

We count the frequency of each issue phrase for every model. This helps answer questions such as:

- Did generated posters reduce low-contrast cases?
- Did they introduce clutter?
- Did text-related issues become less frequent?
- Did whitespace/layout problems increase?

The analysis keeps the issue labels exactly as they appear in the CSVs.


In [ ]:
from collections import Counter

def issue_counts(df):
    counter = Counter()
    for value in df["issues"].dropna():
        if str(value).strip().lower() in {"", "nan", "none"}:
            continue
        for issue in str(value).split(";"):
            issue = issue.strip()
            if issue:
                counter[issue] += 1
    return counter

issue_table = pd.DataFrame({
    model: pd.Series(issue_counts(data[data.model == model]))
    for model in FILES.keys()
}).fillna(0).astype(int)

issue_table["Total mentions"] = issue_table.sum(axis=1)
issue_table = issue_table.sort_values("Total mentions", ascending=False)

display(issue_table)


In [ ]:
# Issue frequency heatmap
plt.figure(figsize=(12, max(6, len(issue_table) * 0.35)))
sns.heatmap(issue_table.drop(columns="Total mentions"),
            annot=True, fmt="d", cmap="Blues")
plt.title("Issue Frequency by Model")
plt.xlabel("")
plt.ylabel("Issue")
plt.tight_layout()
plt.show()


In [ ]:
# Top issues for each generated model
for model in generated_models:
    print(f"\n=== {model} ===")
    display(
        issue_table[model]
        .sort_values(ascending=False)
        .head(10)
        .to_frame("Count")
    )


## 9. Model-to-Model Consistency

A model can have a higher average score but still be inconsistent.

We therefore examine:
- score standard deviation
- interquartile range
- correlation of each generated model's scores with the original scores
- correlation between generated models

This distinguishes **average performance** from **consistency with the dataset's scoring behavior**.


In [ ]:
# Wide score table
score_wide = (
    data.pivot(index="poster_id", columns="model", values="score")
    .reindex(columns=list(FILES.keys()))
)

consistency = pd.DataFrame({
    "Mean score": score_wide.mean(),
    "Std score": score_wide.std(),
    "IQR": score_wide.quantile(.75) - score_wide.quantile(.25),
    "Correlation with Original": score_wide.corr()["Original"]
}).round(3)

display(consistency)


In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(score_wide.corr(), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation Between Poster Scores Across Models")
plt.tight_layout()
plt.show()


## 10. Per-Poster Model Comparison

For each original poster, compare the four generated scores.

This is useful for finding:
- posters where all models improve
- posters where models disagree strongly
- posters that are difficult to improve
- model-specific strengths and failures

The table is sorted by the spread between the highest and lowest generated scores.


In [ ]:
generated_wide = score_wide[generated_models].copy()

per_poster = generated_wide.copy()
per_poster["Original"] = score_wide["Original"]
per_poster["Generated mean"] = generated_wide.mean(axis=1)
per_poster["Best generated score"] = generated_wide.max(axis=1)
per_poster["Worst generated score"] = generated_wide.min(axis=1)
per_poster["Generated score spread"] = generated_wide.max(axis=1) - generated_wide.min(axis=1)
per_poster["Mean change vs original"] = per_poster["Generated mean"] - per_poster["Original"]

per_poster = per_poster.sort_values("Generated score spread", ascending=False)

display(per_poster.round(2).head(15))


In [ ]:
# Count how often each generated model has the highest score for a poster.
highest_counts = generated_wide.eq(generated_wide.max(axis=1), axis=0).sum()
highest_counts = highest_counts.sort_values(ascending=False)

print("Highest generated score count per poster:")
display(highest_counts.to_frame("Count"))


## 11. Important Diagnostic: Score Change vs Feature Change

This section checks whether score improvements are accompanied by systematic changes in the measurable poster features.

For each generated model, we correlate the **per-poster score change** with each **per-poster feature change**.

A positive correlation means that, in this dataset, larger increases in that feature tend to occur alongside larger score improvements. It does **not** establish causality.


In [ ]:
diagnostic_rows = []

for model in generated_models:
    gen = data[data.model == model].set_index("poster_id")

    score_delta = gen["score"] - original["score"]

    for feature in feature_cols:
        feature_delta_values = gen[feature] - original[feature]
        rho, p = spearmanr(score_delta, feature_delta_values, nan_policy="omit")

        diagnostic_rows.append({
            "Model": model,
            "Feature": feature,
            "rho(score delta, feature delta)": rho,
            "p-value": p
        })

diagnostics = pd.DataFrame(diagnostic_rows)

for model in generated_models:
    print(f"\n=== {model} ===")
    display(
        diagnostics[diagnostics.Model == model]
        .assign(abs_rho=lambda x: x["rho(score delta, feature delta)"].abs())
        .sort_values("abs_rho", ascending=False)
        .drop(columns="abs_rho")
        .round(4)
    )


## 12. Research-Ready Summary

The final table combines the most decision-useful metrics without hiding the underlying distributions.

Interpretation guide:
- **Mean change**: average movement from the original score.
- **Improved %**: proportion of posters whose score increased.
- **Declined %**: proportion whose score decreased.
- **Paired Cohen's d**: standardized magnitude of paired change.
- **Wilcoxon p**: evidence against a zero-median change under the paired test.
- **Std score**: variability of the model's scores.
- **Correlation with Original**: how similarly the model's score pattern behaves across posters.

Do not use the p-value alone to claim a model is "better"; combine magnitude, consistency, feature changes, and issue patterns.


In [ ]:
final_summary = paired_results.copy()

final_summary["Mean score"] = score_summary.loc[generated_models, "mean"]
final_summary["Median score"] = score_summary.loc[generated_models, "median"]
final_summary["Score std"] = score_summary.loc[generated_models, "std"]
final_summary["Correlation with Original"] = consistency.loc[generated_models, "Correlation with Original"]

final_summary = final_summary[
    [
        "Mean score",
        "Median score",
        "Mean change",
        "Median change",
        "Improved %",
        "Declined %",
        "Paired Cohen's d",
        "Wilcoxon p",
        "Score std",
        "Correlation with Original"
    ]
].round(3)

display(final_summary)


## 13. Automatically Generated Findings

This cell prints concise findings from the actual CSV values so the conclusions remain tied to the dataset.

These statements are descriptive and should be checked against the plots before being used in a report/paper.


In [ ]:
# Generate dataset-specific observations
for model in generated_models:
    r = final_summary.loc[model]
    print(
        f"{model}: mean score {r['Mean score']:.2f}, "
        f"mean change {r['Mean change']:+.2f}, "
        f"{r['Improved %']:.1f}% improved and {r['Declined %']:.1f}% declined; "
        f"paired Cohen's d={r['Paired Cohen\'s d']:.2f}, "
        f"Wilcoxon p={r['Wilcoxon p']:.4g}."
    )

print("\nOriginal mean score:", score_summary.loc["Original", "mean"])
print("Original median score:", score_summary.loc["Original", "median"])


## 14. Export Results

The following files are created for use in a report or thesis:
- `model_score_summary.csv`
- `paired_score_changes.csv`
- `feature_means.csv`
- `feature_deltas.csv`
- `issue_frequency.csv`
- `poster_level_comparison.csv`
- `score_feature_correlations.csv`
- `score_delta_feature_delta_correlations.csv`


In [ ]:
OUT = BASE_DIR / "poster_model_comparison_results"
OUT.mkdir(exist_ok=True)

score_summary.to_csv(OUT / "model_score_summary.csv")
paired_results.to_csv(OUT / "paired_score_changes.csv")
feature_means.to_csv(OUT / "feature_means.csv")
feature_delta.to_csv(OUT / "feature_deltas.csv")
issue_table.to_csv(OUT / "issue_frequency.csv")
per_poster.to_csv(OUT / "poster_level_comparison.csv")
corr_df.to_csv(OUT / "score_feature_correlations.csv", index=False)
diagnostics.to_csv(OUT / "score_delta_feature_delta_correlations.csv", index=False)

print(f"Saved results to: {OUT}")


# Suggested Results to Report

For a paper/thesis, the most valuable results from this notebook are:

1. **Paired mean/median score change** relative to the original dataset.
2. **Percentage of posters improved vs declined** for each model.
3. **Wilcoxon signed-rank test + paired Cohen's d** to quantify statistical evidence and effect magnitude.
4. **Feature-level shifts** showing how generation changes contrast, whitespace, text density, edge density, saliency, etc.
5. **Issue-frequency analysis** to show which design problems are reduced or introduced.
6. **Per-poster disagreement** to identify cases where models behave differently.
7. **Score–feature correlations** as descriptive evidence about which measurable characteristics are associated with the evaluation score.

For your final discussion, avoid saying only that one model has the highest average score. The stronger analysis is to explain **whether the score improvement is consistent, statistically supported, what visual features changed, and what issues were reduced or introduced**.
